# TikTok Shop — Step-by-Step Data Pipeline Demo

## แนวทางสาธิตที่อาจารย์มองเห็นทีละขั้น

**วิธีใช้: เปิด Google Colab → Runtime → Run all เพียงครั้งเดียว** ระบบจะรันทุก Stage ตามลำดับ และแสดง Output แยกใต้แต่ละ Cell เพื่อเห็นข้อมูลจริงของแต่ละชั้น

**Run 1:** GitHub CSV → SQLite ER1 → Landing/Raw/Standardized Parquet → DuckDB ER2 → Quality/Lineage

**Run 2:** เพิ่ม/อัปเดต Source ชุด Incremental ลงใน **Database เดิม** แล้วผ่าน Lake และ Warehouse เดิม → ตรวจยอดก่อน/หลัง → รัน Batch ซ้ำเพื่อยืนยันว่าไม่ซ้ำ

**Error Test:** จำลองความผิดพลาด → ตรวจจับ → แก้ → Retry โดยใช้พื้นที่แยกจากธุรกิจหลัก

> ข้อมูลจำลองทั้งหมด ไม่ใช่ TikTok Shop API; ต้นทุนและค่าธรรมเนียมใช้สมมติฐานของโปรเจกต์  
> ทุกครั้งที่สั่ง **Run all** ระบบจะเริ่มพื้นที่ทดลอง `output/main` ใหม่หนึ่งครั้งที่ Setup เท่านั้น แล้ว Run 1 → Run 2 จะใช้ฐานข้อมูลชุดเดียวกัน โดย **ไม่มีการ Reset คั่นระหว่างสอง Run**

> กด **Run all ซ้ำ** = สร้าง Demo ใหม่ครบทั้งกระบวนการจาก Source ชุดเดิม; ส่วน **Run 2** = เพิ่มข้อมูล Batch ใหม่ลงฐานเดิม; และ **Replay** = ส่ง Batch เดิมซ้ำเพื่อตรวจว่าไม่มีข้อมูลซ้ำ

## 00 | Setup — เตรียม Code และ Source จาก GitHub

**ทำอะไร:** ดาวน์โหลด Code และ CSV จาก GitHub (เฉพาะครั้งแรกของ Runtime) และเตรียมพื้นที่เก็บ Output

**กด Runtime → Run all ได้ทันที** หากกด Run all ซ้ำ Setup จะล้างเฉพาะ `output/main` ใน Colab ก่อนสร้างผลใหม่ ไม่กระทบข้อมูล Source บน GitHub หรือ Snapshot บน Google Drive


In [ ]:
#@title 00 — เตรียมโปรเจกต์
REPO_URL = "https://github.com/KankaweeK356/tiktok-shop-pipeline-demo"
from pathlib import Path
import os, sys, subprocess, importlib, sqlite3
from IPython.display import display

project_name = "TikTok_Shop_Pipeline_Demo"
if (Path.cwd() / "pipeline" / "orchestrator.py").is_file():
    ROOT = Path.cwd()
else:
    ROOT = Path("/content") / project_name if Path("/content").is_dir() else Path.cwd() / project_name
    if not ROOT.exists():
        print("กำลังดาวน์โหลด Source และ Code จาก GitHub ...")
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)])
    else:
        print("ใช้ไฟล์โปรเจกต์ที่มีใน Runtime นี้ ไม่ git pull ซ้ำ")
required = ["requirements.txt", "pipeline/orchestrator.py", "pipeline/transform.py",
            "source/initial/Fact_Orders.csv", "source/incremental/Fact_Orders.csv"]
missing = [p for p in required if not (ROOT / p).is_file()]
if missing:
    raise RuntimeError("โครงสร้างโปรเจกต์ไม่ครบ: " + ", ".join(missing))
try:
    import duckdb, pyarrow, pandas, matplotlib, nbformat
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r",
                           str(ROOT / "requirements.txt")])
root_path = str(ROOT.resolve())
sys.path = [p for p in sys.path if p != root_path]
sys.path.insert(0, root_path)
for module_name in list(sys.modules):
    if module_name == "pipeline" or module_name.startswith("pipeline."):
        del sys.modules[module_name]
importlib.invalidate_caches()

from pipeline.orchestrator import run_batch, reset
from pipeline.demo_tests import signature, error_suite, performance_suite
from pipeline.quality import verify, lineage
from pipeline.business import business_report
from pipeline.publication import export_github_reports, EvidenceNotReady
import pandas as pd
import duckdb
import matplotlib.pyplot as plt

WORKSPACE = ROOT / "output" / "main"
# Run all always starts a fresh, reproducible demo. The reset runs ONCE
# here, before Run 1. Run 2 updates the same ER1/ER2 databases; no reset
# happens between batches. Never touch GitHub source or Drive snapshots.
reset(WORKSPACE)
print("✅ เริ่ม Demo ใหม่ — ล้างเฉพาะผลเก่า output/main ใน Colab")
print("Run 1 และ Run 2 จะใช้ Database เดิมต่อเนื่องกัน ไม่มีการ Reset คั่น")
print("✅ พร้อมทดลอง | โปรเจกต์:", ROOT)
print("Source เริ่มต้น:", len(pd.read_csv(ROOT/"source/initial/Fact_Orders.csv")), "Orders")
print("Source เพิ่มเติม:", len(pd.read_csv(ROOT/"source/incremental/Fact_Orders.csv")), "แถวคำสั่งซื้อ")
print("ผลลัพธ์จะอยู่ที่:", WORKSPACE)


## RUN 1 — Initial Load

นำข้อมูลตั้งต้นจาก GitHub CSV เข้า Pipeline ทีละ Stage และแสดงผลของแต่ละชั้น

### 01 | Source CSV

**ทำอะไร:** เปิดไฟล์ข้อมูลต้นทางเพื่อดูขนาดและข้อมูลตัวอย่างก่อนนำเข้า

In [ ]:
#@title 01 — ตรวจ Source Data (Initial)
initial_dir = ROOT / "source" / "initial"
source_orders = pd.read_csv(initial_dir / "Fact_Orders.csv")
source_items = pd.read_csv(initial_dir / "Fact_Order_Items.csv")
print("✅ Source CSV:", len(list(initial_dir.glob("*.csv"))), "ไฟล์")
print("Orders:", len(source_orders), "| Order Items:", len(source_items))
display(source_orders.head(4))


### 02 | SQLite — Business Database (ER 1)

**ทำอะไร:** Ingest Source CSV เข้าฐานข้อมูล ER 1 แสดงจำนวน Orders และ Items ใน SQLite

In [ ]:
#@title 02 — โหลดข้อมูลเข้าฐานข้อมูล ER 1
stage = run_batch(WORKSPACE, "initial", stop_after="ingest")
with sqlite3.connect(WORKSPACE / "er1.sqlite") as con:
    er1_counts = pd.read_sql_query("""SELECT 'Fact_Orders' AS Table_Name, COUNT(*) AS Records FROM Fact_Orders
      UNION ALL SELECT 'Fact_Order_Items',COUNT(*) FROM Fact_Order_Items
      UNION ALL SELECT 'Dim_SKU',COUNT(*) FROM Dim_SKU""", con)
    er1_example = pd.read_sql_query("SELECT Order_ID,Current_Status,Paid_At FROM Fact_Orders LIMIT 4",con)
display(er1_counts)
display(er1_example)
print("✅ Ingest สำเร็จ | ผ่านขั้นตอน:", ", ".join(stage["completed"]))


### 03 | Data Lake — Landing และ Raw

**ทำอะไร:** เก็บสำเนา Source และ Extract จาก SQLite เป็น Raw Parquet พร้อมแสดงจำนวนไฟล์

In [ ]:
#@title 03 — สร้าง Landing และ Raw Parquet
stage = run_batch(WORKSPACE, "initial", stop_after="extract")
landing = WORKSPACE / "lake" / "landing" / "initial"
raw = WORKSPACE / "lake" / "raw" / "initial"
raw_orders = pd.read_parquet(raw / "Fact_Orders.parquet")
print("✅ Landing:", len(list(landing.glob("*.csv"))), "ไฟล์ CSV")
print("✅ Raw:", len(list(raw.glob("*.parquet"))), "ตาราง Parquet")
print("Raw Fact_Orders:", len(raw_orders), "แถว")
display(raw_orders[["Order_ID", "Current_Status", "Paid_At"]].head(4))


### 04 | Data Lake — Standardized

**ทำอะไร:** Transform ข้อมูลจาก Raw ให้พร้อมเข้าสู่ ER 2 รวมถึงการคำนวณราคาขายและต้นทุน

In [ ]:
#@title 04 — Transform เป็น Standardized Parquet
stage = run_batch(WORKSPACE, "initial", stop_after="transform")
standard = WORKSPACE / "lake" / "standardized" / "initial"
sales_std = pd.read_parquet(standard / "Fact_Sales_Order_Item.parquet")
print("✅ Standardized:", len(list(standard.glob("*.parquet"))), "ตาราง Parquet")
print("Sales Item ที่เตรียมโหลด:", len(sales_std), "แถว")
display(sales_std[["Source_Order_Item_ID", "Net_Revenue", "COGS", "Platform_Fee_Cost"]].head(4))


### 05 | DuckDB — Data Warehouse (ER 2)

**ทำอะไร:** Load Fact และ Dimension พร้อมสร้าง Daily Mart ใน Warehouse

In [ ]:
#@title 05 — โหลดเข้าคลังข้อมูล ER 2
stage = run_batch(WORKSPACE, "initial", stop_after="load")
with duckdb.connect(str(WORKSPACE / "er2.duckdb"), read_only=True) as con:
    dw_counts = con.execute("""SELECT 'Fact_Sales_Order_Item' AS Table_Name, COUNT(*) AS Records FROM Fact_Sales_Order_Item
      UNION ALL SELECT 'Dim_Product',COUNT(*) FROM Dim_Product
      UNION ALL SELECT 'Mart_Product_Group_Daily_Performance',COUNT(*) FROM Mart_Product_Group_Daily_Performance""").df()
    dw_example = con.execute("""SELECT Source_Order_Item_ID, Net_Revenue, COGS, Affiliate_Cost
      FROM Fact_Sales_Order_Item LIMIT 4""").df()
display(dw_counts)
display(dw_example)
print("✅ Warehouse โหลดสำเร็จ")


### 06 | Functional Validation และ Lineage

**ทำอะไร:** เปรียบเทียบ ER1 กับ ER2 ตรวจยอดและตามที่มา Order Item ได้

In [ ]:
#@title 06 — ตรวจยอดและ Lineage หลัง Run 1
initial_result = run_batch(WORKSPACE, "initial")
INITIAL_SIGNATURE = signature(WORKSPACE)
initial_checks = pd.read_csv(WORKSPACE / "reports" / "functional_initial.csv")
print("✅ Run 1:", initial_result["status"])
print("Functional:", initial_result["functional_passes"], "ผ่าน |",
      initial_result["functional_fails"], "ไม่ผ่าน")
display(initial_checks[["Check", "Actual", "Expected", "PASS"]].head(10))
print("จำนวน Sales Items:", INITIAL_SIGNATURE["orders"],
      "| Net Revenue:", INITIAL_SIGNATURE["sales"], "บาท")


## RUN 2 — Incremental Load (ไม่ Reset ฐานข้อมูลเดิม)

รอบนี้ใช้ **ER1/ER2 เดิม** (Setup ล้างเฉพาะตอนเริ่ม Run all ไม่ได้ล้างระหว่างสองรอบ) เพิ่มคำสั่งซื้อใหม่ พร้อมอัปเดต Pending → Paid และข้อมูลต้นทุนที่เข้าทีหลัง

### 07 | Source CSV — Incremental

**ทำอะไร:** ตรวจรายการ Source ชุดใหม่ และแยกว่าเพิ่ม Order ใหม่หรือแก้สถานะ Order เดิม

In [ ]:
#@title 07 — ตรวจ Source Data รอบ Incremental
incremental_dir = ROOT / "source" / "incremental"
incremental_orders = pd.read_csv(incremental_dir / "Fact_Orders.csv")
initial_ids = set(source_orders["Order_ID"].astype(str))
new_orders = incremental_orders[~incremental_orders["Order_ID"].isin(initial_ids)]
changed_orders = incremental_orders[incremental_orders["Order_ID"].isin(initial_ids)]
print("✅ Incremental Orders ใหม่:", len(new_orders))
print("✅ Order เดิมที่มีการอัปเดต:", len(changed_orders))
display(incremental_orders[["Order_ID", "Current_Status", "Paid_At"]].head(6))


### 08 | SQLite ER 1 — Upsert

**ทำอะไร:** Ingest ข้อมูลใหม่ลง Database เดิม ไม่ใช่สร้างฐานใหม่

In [ ]:
#@title 08 — อัปเดตฐานข้อมูล ER 1
stage = run_batch(WORKSPACE, "incremental", stop_after="ingest")
with sqlite3.connect(WORKSPACE / "er1.sqlite") as con:
    er1_after = pd.read_sql_query("""SELECT COUNT(*) AS Orders_Total,
      SUM(CASE WHEN Current_Status IN ('Paid','Shipped','Delivered') THEN 1 ELSE 0 END) AS Paid_Orders
      FROM Fact_Orders""", con)
display(er1_after)
print("✅ ER1 เพิ่ม/อัปเดตข้อมูลเสร็จ; ไม่มีการ Reset")


### 09 | Data Lake — Incremental Landing/Raw

**ทำอะไร:** สร้าง Source Receipt และ Raw Snapshot ของ ER1 หลังเพิ่มข้อมูล

In [ ]:
#@title 09 — สร้าง Landing และ Raw รอบใหม่
stage = run_batch(WORKSPACE, "incremental", stop_after="extract")
landing2 = WORKSPACE / "lake" / "landing" / "incremental"
raw2 = WORKSPACE / "lake" / "raw" / "incremental"
print("✅ Landing Incremental:", len(list(landing2.glob("*.csv"))), "ไฟล์")
print("✅ Raw Snapshot:", len(list(raw2.glob("*.parquet"))), "ตาราง")
print("Raw Orders ทั้งหมด:", len(pd.read_parquet(raw2/"Fact_Orders.parquet")), "แถว")


### 10 | Data Lake — Incremental Standardized

**ทำอะไร:** แปลงเฉพาะรายการ Fact ที่เปลี่ยน แสดงจำนวน Delta ที่เตรียมโหลด

In [ ]:
#@title 10 — Transform ข้อมูลที่เปลี่ยน
stage = run_batch(WORKSPACE, "incremental", stop_after="transform")
std_delta = pd.read_parquet(WORKSPACE/"lake"/"standardized"/"incremental"/"Fact_Sales_Order_Item.parquet")
print("✅ Changed Sales Items:", len(std_delta), "แถว")
display(std_delta[["Source_Order_Item_ID","Net_Revenue","COGS"]].head(4))


### 11 | DuckDB ER 2 — Incremental Load

**ทำอะไร:** Upsert Delta ไปที่ Warehouse เดิม แล้วเปรียบเทียบจำนวนรายการและยอดขาย

In [ ]:
#@title 11 — อัปเดต Warehouse เดิม
stage = run_batch(WORKSPACE, "incremental", stop_after="load")
after_load = signature(WORKSPACE)
display(pd.DataFrame([
    {"จุดเปรียบเทียบ":"หลัง Run 1", "Sales Items":INITIAL_SIGNATURE["orders"], "Net Revenue":INITIAL_SIGNATURE["sales"]},
    {"จุดเปรียบเทียบ":"หลัง Run 2", "Sales Items":after_load["orders"], "Net Revenue":after_load["sales"]}
]))
assert after_load["orders"] > INITIAL_SIGNATURE["orders"], "Incremental ไม่เพิ่มรายการขาย"
print("✅ Warehouse Updated")


### 12 | Validate — Run 2

**ทำอะไร:** ตรวจยอด ER1/ER2 และ Lineage หลังการอัปเดต

In [ ]:
#@title 12 — ตรวจคุณภาพข้อมูลหลัง Run 2
incremental_result = run_batch(WORKSPACE, "incremental")
final_check = verify(WORKSPACE/"er1.sqlite", WORKSPACE/"er2.duckdb",
                     WORKSPACE/"reports", "final")
lineage_result = lineage(WORKSPACE/"er1.sqlite", WORKSPACE/"er2.duckdb",
                         "final", WORKSPACE/"lake"/"control",
                         WORKSPACE/"reports", limit=6)
print("✅ Run 2:", incremental_result["status"])
print("✅ Functional Checks:", final_check["passed"], "/", final_check["checks"])
print("✅ Lineage:", lineage_result["traced_items"], "รายการ")
display(pd.read_csv(lineage_result["file"]).head(3))


## REPLAY — รัน Incremental Batch เดิมซ้ำ

**จุดประสงค์:** พิสูจน์ว่า Retry/Replay **ไม่สร้างยอดและรายการซ้ำ** ซึ่งต่างจากการรับข้อมูลใหม่ใน Run 2

In [ ]:
#@title 13 — Replay Batch เดิมเพื่อตรวจ Idempotency
before_replay = signature(WORKSPACE)
replayed = run_batch(WORKSPACE, "incremental")
after_replay = signature(WORKSPACE)
display(pd.DataFrame([
    {"ช่วง":"ก่อน Replay", "Sales Items":before_replay["orders"], "Net Revenue":before_replay["sales"]},
    {"ช่วง":"หลัง Replay", "Sales Items":after_replay["orders"], "Net Revenue":after_replay["sales"]}
]))
assert before_replay == after_replay, "❌ พบข้อมูลซ้ำหลัง Replay"
print("✅ Idempotency PASS — รัน Batch เดิมซ้ำ ตัวเลขไม่เปลี่ยน")


## ERROR TEST — Fail → Detect → Fix → Retry → Success

**ทำอะไร:** เลือก Case แล้วจำลอง Error จริงในพื้นที่ทดสอบที่แยกจากข้อมูลหลัก ระบบแก้และ Retry จากขั้นที่ผิดให้ดู

In [ ]:
#@title 14 — เลือก Error Case และดูผล Recovery
ERROR_CASE = "T01" #@param ["E01","E02","T01","T02","L01","L02"]
case_names = {
    "E01":"SKU ไม่อยู่ใน Master", "E02":"ไฟล์ Source ขาด",
    "T01":"Voucher ผิดเงื่อนไข", "T02":"Campaign วันทับซ้อน",
    "L01":"Warehouse Load Fail", "L02":"รัน Batch ซ้ำ"
}
chosen = error_suite(ROOT / "output" / "test", cases=[ERROR_CASE])
result_view = chosen[["case", "stage", "detected", "retry_from", "before", "after", "passed"]].copy()
result_view.insert(1, "คำอธิบาย", result_view["case"].map(case_names))
display(result_view)
assert chosen["passed"].all(), "Error Recovery ไม่ผ่าน"
print("✅ Error Case", ERROR_CASE, ": ตรวจพบและกู้คืนสำเร็จ")


## FINAL TEST — Performance และ Business KPI

ขั้นตอนเสริมสำหรับเก็บหลักฐานส่งอาจารย์: ทดสอบ Error ครบ 6 กรณี วัด Latency/Throughput และสรุปรายงานจริง

In [ ]:
#@title 15 — ทดสอบครบชุด และสรุป Business Results
RUN_FULL_TEST = True #@param {type:"boolean"}
if RUN_FULL_TEST:
    six = error_suite(ROOT / "output" / "test", cases=None)
    assert len(six) == 6 and six["passed"].all()
    print("✅ Error Recovery:", int(six["passed"].sum()), "/6")
    perf = performance_suite(ROOT / "output" / "performance", sizes=(100,300))
    display(perf[["initial_source_orders","latency_seconds",
                  "throughput_paid_items_per_second"]].rename(columns={
                      "initial_source_orders":"Orders",
                      "latency_seconds":"Latency (s)",
                      "throughput_paid_items_per_second":"Throughput (items/s)"}))
    monthly, groups = business_report(WORKSPACE/"er2.duckdb", WORKSPACE/"reports")
    display(monthly[["Year_Month","Net_Revenue","Contribution_Profit",
                     "Profit_Protection_Status"]].rename(columns={
                         "Year_Month":"เดือน",
                         "Net_Revenue":"ยอดขายสุทธิ",
                         "Contribution_Profit":"กำไรหลังต้นทุน",
                         "Profit_Protection_Status":"สถานะเกณฑ์กำไร"}))
    print("✅ รายงานทั้งหมดอยู่:", WORKSPACE/"reports")
    try:
        evidence = export_github_reports(ROOT)
        print("✅ ZIP รายงานสำหรับ GitHub:", evidence["zip_path"])
    except EvidenceNotReady as exc:
        print("รายงาน GitHub ยังไม่ครบ:", exc)
else:
    print("ข้าม Full Test ได้; ผล Run 1, Run 2, Replay และ Error Case ยังใช้งานได้")


## SAVE — Google Drive (ไม่บังคับ)

หากต้องการเก็บ SQLite, DuckDB, Parquet และ Reports ถาวรให้ติ๊ก Save; ปิดไว้โดยปกติ จึงไม่มีการขอเข้าถึง Google Drive

In [ ]:
#@title 16 — บันทึกผลลง Drive (ไม่บังคับ)
SAVE_TO_DRIVE = False #@param {type:"boolean"}
if SAVE_TO_DRIVE:
    from google.colab import drive
    from pipeline.drive_storage import save_to_drive
    drive.mount("/content/drive")
    saved = save_to_drive(ROOT, Path("/content/drive/MyDrive"))
    print("✅ บันทึกลง Google Drive:", saved["path"])
else:
    print("ไม่เชื่อม Drive — ผลทั้งหมดอยู่ใน Colab Runtime")
    print("ต้องการบันทึกถาวร: ติ๊ก SAVE_TO_DRIVE แล้วรันเฉพาะ Cell นี้")
